In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import Row

In [ ]:
spark = SparkSession.builder.appName("MindboxTest").getOrCreate()
spark

### Через промежуточную таблицу

In [ ]:
products_data = [
    Row(product_id=1, product_name="Apple"),
    Row(product_id=2, product_name="Banana"),
    Row(product_id=3, product_name="Orange"),
]

In [ ]:
products = spark.createDataFrame(products_data)

In [ ]:
products.show()

+----------+------------+
|product_id|product_name|
+----------+------------+
|         1|       Apple|
|         2|      Banana|
|         3|      Orange|
+----------+------------+



In [ ]:
categories_data = [
    Row(category_id=10, category_name="Fruit"),
    Row(category_id=20, category_name="Yellow Food"),
]

categories = spark.createDataFrame(categories_data)

In [ ]:
categories.show()

+-----------+-------------+
|category_id|category_name|
+-----------+-------------+
|         10|        Fruit|
|         20|  Yellow Food|
+-----------+-------------+



In [ ]:
product_categories_data = [
    Row(product_id=1, category_id=10),
    Row(product_id=2, category_id=10),
    Row(product_id=2, category_id=20),
]

product_categories = spark.createDataFrame(product_categories_data)

In [ ]:
product_categories.show()

+----------+-----------+
|product_id|category_id|
+----------+-----------+
|         1|         10|
|         2|         10|
|         2|         20|
+----------+-----------+



In [ ]:
def get_products_with_categories(products, categories, product_categories):
    prod_with_cat = products.join(product_categories, on='product_id', how='left')

    prod_with_cat = prod_with_cat.join(categories, on='category_id', how='left')

    return prod_with_cat.select('product_name', 'category_name')

result = get_products_with_categories(products, categories, product_categories)
result.show()

+------------+-------------+
|product_name|category_name|
+------------+-------------+
|      Orange|         NULL|
|       Apple|        Fruit|
|      Banana|        Fruit|
|      Banana|  Yellow Food|
+------------+-------------+



### Без промежуточной таблицы

In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import explode, col, lit, size

In [ ]:
products_data = [
    Row(product_id=1, product_name="Apple",  category_ids=[10]),
    Row(product_id=2, product_name="Banana", category_ids=[10, 20]),
    Row(product_id=3, product_name="Orange", category_ids=[]),
]
categories_data = [
    Row(category_id=10, category_name="Fruit"),
    Row(category_id=20, category_name="Yellow Food"),
]

products_df   = spark.createDataFrame(products_data)
categories_df = spark.createDataFrame(categories_data)

In [ ]:
products_df.show()

+----------+------------+------------+
|product_id|product_name|category_ids|
+----------+------------+------------+
|         1|       Apple|        [10]|
|         2|      Banana|    [10, 20]|
|         3|      Orange|          []|
+----------+------------+------------+



In [ ]:
categories_df.show()

+-----------+-------------+
|category_id|category_name|
+-----------+-------------+
|         10|        Fruit|
|         20|  Yellow Food|
+-----------+-------------+



In [ ]:
def get_products_with_categories2(
    products_df: DataFrame,
    categories_df: DataFrame
) -> DataFrame:
    exploded = products_df.filter(size(col("category_ids")) > 0) \
        .withColumn("category_id", explode(col("category_ids")))

    joined = exploded.join(
        categories_df.select("category_id", "category_name"),
        on="category_id",
        how="left"
    ).select(
        col("product_id"),
        col("product_name"),
        col("category_name")
    )

    no_categories = products_df.filter(size(col("category_ids")) == 0) \
        .select(
            col("product_id"),
            col("product_name"),
            lit(None).cast("string").alias("category_name")
        )

    return joined.unionByName(no_categories)

result = get_products_with_categories2(products_df, categories_df)
result.show()

+----------+------------+-------------+
|product_id|product_name|category_name|
+----------+------------+-------------+
|         1|       Apple|        Fruit|
|         2|      Banana|        Fruit|
|         2|      Banana|  Yellow Food|
|         3|      Orange|         NULL|
+----------+------------+-------------+

